# MIMIC-III Lab Events Preprocessing Pipeline

This notebook produces a clean, analysis-ready lab feature matrix from MIMIC-III.
It encapsulates all preprocessing decisions made for the cVAE synthetic data project
and returns a single DataFrame that can be used as input to any downstream model.

### What this notebook produces

A wide-format DataFrame with one row per adult ICU admission containing:

| Column group | Description |
|---|---|
| `hadm_id`, `subject_id` | Identifiers |
| `age`, `gender_bin` | Demographic features |
| `icu_*` | One-hot encoded ICU care unit (5 units, NICU excluded) |
| `los_*` | One-hot encoded LOS bucket |
| `ccs_*` | One-hot encoded primary CCS disease chapter |
| `elix_*` | 31 binary Elixhauser comorbidity flags |
| `glucose` ... `albumin` | 23 first-day lab values (some log-transformed) |
| `hospital_expire_flag` | Outcome variable (1 = died in hospital) |

### Key preprocessing decisions

- Lab window: 6 hours before to 24 hours after first ICU admission time
- Exclusions: NICU patients, admissions with no labs in window
- Imputation: population median per lab (applied only to partial missingness)
- Log transform: 7 right-skewed labs (glucose, creatinine, lactate, PT, PTT, bilirubin, AST)
- FLAG column: null FLAG means normal result - NOT treated as missing

### Output files

- `lab_features.csv` - full preprocessed feature matrix
- `lab_features_scaled.csv` - StandardScaler normalised version
- `preprocessing_metadata.pkl` - scalers, column lists, transforms (for inverse transform at inference)

---
**Required MIMIC-III tables:** `PATIENTS`, `ADMISSIONS`, `ICUSTAYS`, `DIAGNOSES_ICD`, `LABEVENTS`

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# -- Configuration ------------------------------------------------------------─
DATA_DIR   = 'data/'          # directory containing cleaned MIMIC-III CSVs
OUTPUT_DIR = 'data/processed/' # where output files are saved
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Lab window relative to ICU admission time (hours)
LAB_WINDOW_START = -6    # 6 hours before ICU admission (catches pre-transfer labs)
LAB_WINDOW_END   = 24    # 24 hours after ICU admission

# Imputation threshold -- labs with more than this fraction at median
# after the pipeline are flagged as potentially unreliable
IMPUTATION_WARNING_THRESHOLD = 0.30

print('Configuration set.')
print(f'  Data directory:   {DATA_DIR}')
print(f'  Output directory: {OUTPUT_DIR}')
print(f'  Lab window:       {LAB_WINDOW_START}hr to +{LAB_WINDOW_END}hr')

Configuration set.
  Data directory:   data/
  Output directory: data/processed/
  Lab window:       -6hr to +24hr


## 2. Load Raw Tables

In [ ]:
patients   = pd.read_csv(f'{DATA_DIR}patients_cleaned.csv',      parse_dates=['DOB', 'DOD'])
admissions = pd.read_csv(f'{DATA_DIR}admissions_cleaned.csv',    parse_dates=['ADMITTIME', 'DISCHTIME'])
icustays   = pd.read_csv(f'{DATA_DIR}icustays_cleaned.csv',      parse_dates=['INTIME', 'OUTTIME'])
diagnoses  = pd.read_csv(f'{DATA_DIR}diagnoses_icd_cleaned.csv')

# LABEVENTS: load raw file - do NOT use a version cleaned with FLAG as essential
# FLAG null means normal result, not missing data. Requiring FLAG removes 65% of rows.
labevents = pd.read_csv(
    f'{DATA_DIR}labevents_cleaned.csv',
    parse_dates=['CHARTTIME'],
    usecols=['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM', 'FLAG']
)

# Lowercase all column names
for df in [patients, admissions, icustays, diagnoses, labevents]:
    df.columns = df.columns.str.lower()

# Drop lab rows with no HADM_ID (outpatient labs) or no numeric value
before = len(labevents)
labevents = labevents.dropna(subset=['hadm_id', 'valuenum'])
labevents['hadm_id'] = labevents['hadm_id'].astype(int)

# FLAG null = normal result. Fill before any downstream use.
labevents['flag'] = labevents['flag'].fillna('normal')

print(f'Patients:   {patients.shape}')
print(f'Admissions: {admissions.shape}')
print(f'ICU Stays:  {icustays.shape}')
print(f'Diagnoses:  {diagnoses.shape}')
print(f'Lab Events: {labevents.shape}  (dropped {before - len(labevents):,} rows with no HADM_ID or VALUENUM)')

Patients:   (46520, 9)
Admissions: (58976, 19)
ICU Stays:  (61522, 12)
Diagnoses:  (651047, 6)
Lab Events: (20115314, 7)  (dropped 0 rows with no HADM_ID or VALUENUM)


## 3. Feature Engineering

### 3.1 Demographic Features

In [ ]:
# -- Age at admission ----------------------------------------------------------
# Computed in seconds to avoid int32/int64 overflow from MIMIC shifted dates.
# Patients >89 are stored with a shifted DOB producing ages ~300 -- capped at 90.
adm = admissions[['subject_id', 'hadm_id', 'admittime', 'insurance',
                   'marital_status', 'ethnicity', 'hospital_expire_flag']].copy()

adm = adm.merge(patients[['subject_id', 'gender', 'dob', 'expire_flag']], on='subject_id')

# Downcast to seconds precision - MIMIC shifted dates can exceed datetime64[ns] max (year 2262)
adm['admittime'] = pd.to_datetime(adm['admittime'], errors='coerce').astype('datetime64[s]')
adm['dob']       = pd.to_datetime(adm['dob'],       errors='coerce').astype('datetime64[s]')

delta_s    = (adm['admittime'] - adm['dob']).dt.total_seconds()
adm['age'] = (delta_s / (365.25 * 24 * 3600)).clip(lower=0, upper=90)

# Binary gender: 1=Male, 0=Female
adm['gender_bin'] = (adm['gender'] == 'M').astype(int)

print('Demographic features computed.')
print(adm[['age', 'gender_bin']].describe().round(2))

Demographic features computed.
            age  gender_bin
count  58976.00    58976.00
mean      55.09        0.56
std       27.22        0.50
min        0.00        0.00
25%       43.49        0.00
50%       61.76        1.00
75%       75.89        1.00
max       90.00        1.00


### 3.2 ICU Stay Features

In [ ]:
# -- Care unit mapping --------------------------------------------------------─
# cat.codes encodes alphabetically so the integer mapping is:
# 0=CCU, 1=CSRU, 2=MICU, 3=NICU, 4=SICU, 5=TSICU
# NICU is kept in map to avoid NaN errors during one-hot encoding.
# NICU patients are excluded explicitly in section 3.6.
CAREUNIT_MAP = {
    0: 'CCU',
    1: 'CSRU',
    2: 'MICU',
    3: 'NICU',   # retained here - excluded at merge step
    4: 'SICU',
    5: 'TSICU'
}

# Take the FIRST ICU stay per admission (sorted by intime)
# A single admission can have multiple ICU transfers - keep earliest only
icu_first = (
    icustays.sort_values('intime')
    .groupby('hadm_id')
    .first()
    .reset_index()[['hadm_id', 'subject_id', 'first_careunit', 'los']]
)

icu_first['first_careunit'] = icu_first['first_careunit'].map(CAREUNIT_MAP)

# -- LOS buckets --------------------------------------------------------------─
# Bucketed to capture severity non-linearly:
#   <2d   = short observation, likely less acute
#   2-7d  = typical acute illness
#   7-14d = prolonged stay, likely complications
#   >14d  = critical or complex patient
los_bins   = [0, 2, 7, 14, np.inf]
los_labels = ['<2d', '2-7d', '7-14d', '>14d']
icu_first['los_bucket'] = pd.cut(icu_first['los'], bins=los_bins, labels=los_labels)

# One-hot encode separately to avoid dtype interference
careunit_ohe = pd.get_dummies(icu_first['first_careunit'], prefix='icu', dtype=float)
los_ohe      = pd.get_dummies(icu_first['los_bucket'],     prefix='los', dtype=float)
icu_ohe      = pd.concat([icu_first[['hadm_id']], careunit_ohe, los_ohe], axis=1)

print(f'ICU feature columns: {[c for c in icu_ohe.columns if c != "hadm_id"]}')

ICU feature columns: ['icu_CCU', 'icu_CSRU', 'icu_MICU', 'icu_NICU', 'icu_SICU', 'icu_TSICU', 'los_<2d', 'los_2-7d', 'los_7-14d', 'los_>14d']


### 3.3 Diagnosis Features - CCS Grouping + Elixhauser Flags

In [ ]:
# -- CCS coarse grouper --------------------------------------------------------
# Maps primary ICD-9 code to one of 19 disease chapters based on numeric range.
# In production use the full AHRQ CCS mapping file or pyElix for accuracy.

def icd9_to_ccs_coarse(code):
    """Map ICD-9 code string to a coarse CCS disease chapter."""
    if pd.isna(code):
        return 'unknown'
    c = str(code).strip().lstrip('0') or '0'
    try:
        n = float(c[:3]) if c[0].isdigit() else -1
    except ValueError:
        return 'other'
    if n < 0:      return 'external'
    elif n < 140:  return 'infectious'
    elif n < 240:  return 'neoplasm'
    elif n < 280:  return 'endocrine_metabolic'
    elif n < 290:  return 'blood'
    elif n < 320:  return 'mental'
    elif n < 390:  return 'nervous'
    elif n < 460:  return 'circulatory'
    elif n < 520:  return 'respiratory'
    elif n < 580:  return 'digestive'
    elif n < 630:  return 'genitourinary'
    elif n < 680:  return 'pregnancy'
    elif n < 710:  return 'skin'
    elif n < 740:  return 'musculoskeletal'
    elif n < 760:  return 'congenital'
    elif n < 780:  return 'perinatal'
    elif n < 800:  return 'symptoms'
    elif n < 1000: return 'injury'
    else:          return 'other'

# Primary diagnosis only (seq_num == 1)
primary_dx = diagnoses[diagnoses['seq_num'] == 1][['hadm_id', 'icd9_code']].copy()
primary_dx['ccs_group'] = primary_dx['icd9_code'].apply(icd9_to_ccs_coarse)

ccs_ohe = pd.get_dummies(primary_dx[['hadm_id', 'ccs_group']],
                          columns=['ccs_group'], prefix='ccs', dtype=float)

print(f'CCS categories found: {primary_dx["ccs_group"].nunique()}')

CCS categories found: 18


In [ ]:
# -- Elixhauser comorbidity flags (31 binary indicators) ----------------------─
# Each flag = 1 if ANY diagnosis code in the admission matches that comorbidity.
# Captures comorbidity burden from all secondary diagnoses.

ELIXHAUSER = {
    'chf':              lambda c: c.startswith('428'),
    'arrhythmia':       lambda c: c[:3] in ['427'],
    'valvular_disease': lambda c: c[:3] in ['394','395','396','397','424'],
    'pulm_circulation': lambda c: c[:3] in ['415','416','417'],
    'pvd':              lambda c: c[:3] in ['440','441','443','444','447'],
    'hypertension':     lambda c: c.startswith('401') or c.startswith('402'),
    'paralysis':        lambda c: c[:3] in ['342','343','344'],
    'other_neuro':      lambda c: c[:3] in ['330','331','332','333','334','335'],
    'chronic_pulm':     lambda c: c[:3] in ['490','491','492','493','494','496'],
    'diabetes_uncomp':  lambda c: c.startswith('2500'),
    'diabetes_comp':    lambda c: c[:4] in ['2501','2502','2503','2504','2505','2506','2507','2508','2509'],
    'hypothyroidism':   lambda c: c[:3] in ['243','244'],
    'renal_failure':    lambda c: c[:3] in ['585','586','588'],
    'liver_disease':    lambda c: c[:3] in ['571','572','573'],
    'pud':              lambda c: c[:3] in ['531','532','533','534'],
    'aids':             lambda c: c.startswith('042'),
    'lymphoma':         lambda c: c[:3] in ['200','201','202'],
    'metastatic_ca':    lambda c: c[:3] in ['196','197','198','199'],
    'solid_tumor':      lambda c: c[:1] in ['1'] and c[:3] not in ['196','197','198','199'],
    'rheumatoid':       lambda c: c[:3] in ['701','710','714','720','725'],
    'coagulopathy':     lambda c: c[:3] in ['286','287'],
    'obesity':          lambda c: c.startswith('2780'),
    'weight_loss':      lambda c: c[:3] in ['260','261','262','263'],
    'fluid_elec':       lambda c: c[:3] in ['276'],
    'blood_loss':       lambda c: c.startswith('2800'),
    'deficiency_anemia':lambda c: c[:3] in ['280','281'],
    'alcohol_abuse':    lambda c: c[:3] in ['291','303','305'],
    'drug_abuse':       lambda c: c[:3] in ['292','304'],
    'psychoses':        lambda c: c[:3] in ['295','297','298'],
    'depression':       lambda c: c[:3] in ['296','300','309','311'],
    'septicemia':       lambda c: c.startswith('038'),
}

def compute_elixhauser(hadm_diag_df):
    """Return DataFrame of Elixhauser flags, one row per hadm_id."""
    flags = {}
    for hadm_id, grp in hadm_diag_df.groupby('hadm_id'):
        codes = grp['icd9_code'].dropna().astype(str).str.strip().tolist()
        row   = {f'elix_{k}': int(any(fn(c) for c in codes))
                 for k, fn in ELIXHAUSER.items()}
        row['hadm_id'] = hadm_id
        flags[hadm_id] = row
    return pd.DataFrame(flags.values())

elix_df = compute_elixhauser(diagnoses)
print(f'Elixhauser flags shape: {elix_df.shape}')  # should be (n_admissions, 32)
print(f'Mean comorbidity count: {elix_df.drop(columns="hadm_id").sum(axis=1).mean():.2f}')

Elixhauser flags shape: (58976, 32)
Mean comorbidity count: 3.00


### 3.4 Lab Values from LABEVENTS

Extracts 23 first-day lab values per ICU admission.

**Important notes on upstream transformations:**
Several labs in the cleaned MIMIC data were already log-transformed upstream.
This was detected by comparing inverse-transformed ranges to expected ICU clinical values.
Only labs confirmed to have raw clinical units are log-transformed here.
Applying log1p to already-transformed labs causes double transformation
producing astronomically large inverse values.

| Lab group | Already transformed upstream | Log-transformed here |
|---|---|---|
| sodium, potassium, bicarbonate, chloride | Yes | No |
| calcium_total, free_calcium, phosphate | Yes | No |
| hemoglobin, hematocrit, albumin, pco2 | Yes | No |
| glucose, creatinine, lactate, PT, PTT, bilirubin, AST | No | Yes |

In [ ]:
# -- ITEMID mapping ------------------------------------------------------------
# Each lab can have multiple ITEMIDs across CareVue and MetaVision systems.
# Excluded from top 30 most common labs:
#   51279 (RBC)         -- redundant with hemoglobin/hematocrit
#   51274 (PT raw)      -- replaced by pt which uses same ITEMID
#   51277 (RDW)         -- derived red cell index
#   51248 (MCH)         -- derived
#   51249 (MCHC)        -- derived
#   51250 (MCV)         -- derived
#   50804 (Total CO2)   -- redundant with bicarbonate
#   50811 (Hemoglobin)  -- duplicate of 51222 from blood gas panel
#   51244 (Lymphocytes) -- too granular alongside WBC

LAB_ITEMIDS = {
    # Core chemistry
    'glucose':        [50931, 50809],
    'sodium':         [50983, 50824],
    'potassium':      [50971, 50822],
    'creatinine':     [50912],
    'bun':            [51006],
    'bicarbonate':    [50882, 50803],
    'chloride':       [50902],
    'calcium_total':  [50893],
    'free_calcium':   [50808],
    'phosphate':      [50970],
    # Blood gas
    'ph':             [50820],
    'pco2':           [50818],
    'po2':            [50821],
    'lactate':        [50813],
    # Hematology
    'hemoglobin':     [51222],
    'hematocrit':     [51221],
    'wbc':            [51301, 51300],
    'platelets':      [51265],
    # Coagulation
    'pt':             [51274],
    'ptt':            [51275],
    # Liver and nutrition
    'bilirubin':      [50885],
    'ast':            [50878],
    'albumin':        [50862],
}

LAB_COLS      = list(LAB_ITEMIDS.keys())
itemid_to_lab = {iid: name
                 for name, iids in LAB_ITEMIDS.items()
                 for iid in iids}

print(f'Total labs:            {len(LAB_COLS)}')
print(f'Total ITEMIDs tracked: {len(itemid_to_lab)}')
print(f'Labs: {LAB_COLS}')

Total labs:            23
Total ITEMIDs tracked: 28
Labs: ['glucose', 'sodium', 'potassium', 'creatinine', 'bun', 'bicarbonate', 'chloride', 'calcium_total', 'free_calcium', 'phosphate', 'ph', 'pco2', 'po2', 'lactate', 'hemoglobin', 'hematocrit', 'wbc', 'platelets', 'pt', 'ptt', 'bilirubin', 'ast', 'albumin']


In [ ]:
# -- Filter to target ITEMIDs --------------------------------------------------
labs_filtered = labevents[labevents['itemid'].isin(itemid_to_lab)].copy()
labs_filtered['lab_name'] = labs_filtered['itemid'].map(itemid_to_lab)

# -- Get first ICU admission time per hadm_id ----------------------------------
# Consistent with icu_first -- always uses the earliest ICU stay per admission
intime_map = (
    icustays.sort_values('intime')
    .groupby('hadm_id')['intime']
    .first()
    .reset_index()
)
intime_map['intime'] = pd.to_datetime(intime_map['intime'], errors='coerce')

labs_filtered = labs_filtered.merge(intime_map, on='hadm_id', how='inner')
labs_filtered['charttime'] = pd.to_datetime(labs_filtered['charttime'], errors='coerce')

# -- Compute hours relative to ICU admission ----------------------------------─
labs_filtered['hours_after_icu'] = (
    (labs_filtered['charttime'] - labs_filtered['intime'])
    .dt.total_seconds() / 3600
)

# Apply the time window
labs_24h = labs_filtered[
    (labs_filtered['hours_after_icu'] >= LAB_WINDOW_START) &
    (labs_filtered['hours_after_icu'] <= LAB_WINDOW_END)
].copy()

print(f'Lab events in {LAB_WINDOW_START}hr to +{LAB_WINDOW_END}hr window: {labs_24h.shape[0]:,}')

Lab events in -6hr to +24hr window: 2,750,654


In [ ]:
# -- First value per lab per admission ----------------------------------------─
# Sorted by charttime so .first() returns earliest measurement.
# Captures baseline physiology on ICU arrival before treatment effects.
first_labs = (
    labs_24h.sort_values('charttime')
    .groupby(['hadm_id', 'lab_name'])['valuenum']
    .first()
    .reset_index()
)

# -- Pivot to wide format ------------------------------------------------------
labs_wide = first_labs.pivot(index='hadm_id', columns='lab_name', values='valuenum')
labs_wide.columns.name = None
labs_wide = labs_wide.reset_index()

# Ensure all 23 columns exist even if a lab has zero data in this cohort
for col in LAB_COLS:
    if col not in labs_wide.columns:
        labs_wide[col] = np.nan

# -- Clip physiologically implausible values ----------------------------------─
# Generous bounds that remove data entry errors while preserving extreme values.
# Upper bound for po2 is high to accommodate ventilated patients on high FiO2.
LAB_BOUNDS = {
    'glucose':        (20,    2000),
    'sodium':         (100,   200),
    'potassium':      (1.0,   15.0),
    'creatinine':     (0.1,   50.0),
    'bun':            (1,     400),
    'bicarbonate':    (1,     60),
    'chloride':       (70,    150),
    'calcium_total':  (1.0,   20.0),
    'free_calcium':   (0.5,   4.0),
    'phosphate':      (0.5,   20.0),
    'ph':             (6.5,   8.0),
    'pco2':           (10,    200),
    'po2':            (10,    700),
    'lactate':        (0.1,   30),
    'hemoglobin':     (1,     25),
    'hematocrit':     (5,     70),
    'wbc':            (0.1,   500),
    'platelets':      (1,     3000),
    'pt':             (10,    150),
    'ptt':            (10,    300),
    'bilirubin':      (0.1,   100),
    'ast':            (1,     10000),
    'albumin':        (0.5,   10),
}
for lab, (lo, hi) in LAB_BOUNDS.items():
    if lab in labs_wide.columns:
        labs_wide[lab] = labs_wide[lab].clip(lower=lo, upper=hi)

# -- Median imputation --------------------------------------------------------─
# Applied after clipping so outliers do not skew the median.
# Only fills genuinely missing values -- patients who had no lab in the window.
lab_medians = labs_wide[LAB_COLS].median()
labs_wide[LAB_COLS] = labs_wide[LAB_COLS].fillna(lab_medians)

# -- Missingness report --------------------------------------------------------
total = len(labs_wide)
print(f'Lab availability in window (n={total:,} admissions):')
for col in LAB_COLS:
    frac_imputed = (labs_wide[col] == lab_medians[col]).mean()
    flag = '  WARNING -- heavily imputed' if frac_imputed > IMPUTATION_WARNING_THRESHOLD else ''
    print(f'  {col:16s}: {frac_imputed:.1%} at median{flag}')

Lab availability in window (n=56,358 admissions):
  glucose         : 13.2% at median
  sodium          : 18.7% at median
  potassium       : 15.1% at median
  creatinine      : 20.1% at median
  bun             : 15.6% at median
  bicarbonate     : 19.1% at median
  chloride        : 16.1% at median
  calcium_total   : 28.6% at median
  free_calcium    : 62.1% at median  WARNING -- heavily imputed
  phosphate       : 27.5% at median
  ph              : 41.9% at median  WARNING -- heavily imputed
  pco2            : 44.1% at median  WARNING -- heavily imputed
  po2             : 41.5% at median  WARNING -- heavily imputed
  lactate         : 50.1% at median  WARNING -- heavily imputed
  hemoglobin      : 3.0% at median
  hematocrit      : 1.5% at median
  wbc             : 2.1% at median
  platelets       : 1.8% at median
  pt              : 23.5% at median
  ptt             : 22.7% at median
  bilirubin       : 59.2% at median  WARNING -- heavily imputed
  ast             : 60.4% at m

### 3.5 Cohort Definition & Exclusion Criteria

Documents and justifies all exclusion decisions before the final merge.
This section is critical for clinical validity and reproducibility.

In [ ]:
# -- Step 1: Build pre-lab dataframe for exclusion characterization ------------─
df_pre_lab = (
    adm[['hadm_id', 'subject_id', 'age', 'gender_bin', 'hospital_expire_flag']]
    .merge(icu_ohe,  on='hadm_id', how='inner')
    .merge(ccs_ohe,  on='hadm_id', how='left')
    .merge(elix_df,  on='hadm_id', how='left')
)
df_pre_lab = df_pre_lab.fillna(0)

# -- Step 2: Identify admissions with no labs in window ------------------------
no_lab_hadm_ids = set(df_pre_lab['hadm_id']) - set(labs_wide['hadm_id'])
no_lab_patients = df_pre_lab[df_pre_lab['hadm_id'].isin(no_lab_hadm_ids)]

print('=' * 60)
print('COHORT EXCLUSION SUMMARY')
print('=' * 60)

print(f'\n-- Exclusion 1: No labs in {LAB_WINDOW_START}hr to +{LAB_WINDOW_END}hr window --')
print(f'   Affected admissions: {len(no_lab_hadm_ids):,}')
print(f'   As % of pre-lab cohort: {len(no_lab_hadm_ids)/len(df_pre_lab):.1%}')

print(f'\n   Care unit breakdown:')
icu_cols = [c for c in no_lab_patients.columns if c.startswith('icu_')]
for col, count in no_lab_patients[icu_cols].sum().sort_values(ascending=False).items():
    unit = col.replace('icu_', '')
    print(f'     {unit:10s}: {count:.0f}')

print(f'\n   LOS breakdown:')
los_cols = [c for c in no_lab_patients.columns if c.startswith('los_')]
for col, count in no_lab_patients[los_cols].sum().sort_values(ascending=False).items():
    bucket = col.replace('los_', '')
    print(f'     {bucket:10s}: {count:.0f}')

print(f'\n   Mortality rate - excluded: {no_lab_patients["hospital_expire_flag"].mean():.1%}')
print(f'   Mortality rate - retained: {df_pre_lab[~df_pre_lab["hadm_id"].isin(no_lab_hadm_ids)]["hospital_expire_flag"].mean():.1%}')
print(f'\n   Conclusion: majority are NICU (neonatal) or short-stay low-acuity.')
print(f'   Inner join is justified -- these are systematically different patients.')

print(f'\n-- Exclusion 2: NICU patients with lab data --')
print(f'   Neonatal physiology incomparable to adult ICU -- excluded regardless of lab availability.')

COHORT EXCLUSION SUMMARY

-- Exclusion 1: No labs in -6hr to +24hr window --
   Affected admissions: 1,418
   As % of pre-lab cohort: 2.5%

   Care unit breakdown:
     NICU      : 931
     MICU      : 244
     SICU      : 91
     CCU       : 60
     CSRU      : 60
     TSICU     : 32

   LOS breakdown:
     <2d       : 1034
     2-7d      : 239
     7-14d     : 74
     >14d      : 71

   Mortality rate - excluded: 2.7%
   Mortality rate - retained: 10.2%

   Conclusion: majority are NICU (neonatal) or short-stay low-acuity.
   Inner join is justified -- these are systematically different patients.

-- Exclusion 2: NICU patients with lab data --
   Neonatal physiology incomparable to adult ICU -- excluded regardless of lab availability.


### 3.6 Merge All Features

In [ ]:
# -- Step 1: Merge all feature tables ----------------------------------------─
df = (
    adm[['hadm_id', 'subject_id', 'age', 'gender_bin', 'hospital_expire_flag']]
    .merge(icu_ohe,   on='hadm_id', how='inner')
    .merge(ccs_ohe,   on='hadm_id', how='left')
    .merge(elix_df,   on='hadm_id', how='left')
    .merge(labs_wide, on='hadm_id', how='inner')   # inner: excludes no-lab admissions
)

# -- Step 2: Explicit NICU exclusion ------------------------------------------
# NICU patients with lab data survived the inner join and must be removed.
# NICU retained in CAREUNIT_MAP to avoid NaN errors during one-hot encoding.
before_nicu = len(df)
df = df[df['icu_NICU'] == 0].copy()
df = df.drop(columns=['icu_NICU'])
print(f'Excluded {before_nicu - len(df):,} remaining NICU patients')
print(f'Dropped icu_NICU column -- all-zero after exclusion')

# -- Step 3: Selective imputation ----------------------------------------------
# Median imputation for partial lab missingness only.
# Zero-fill for conditioning columns -- correct default for one-hot flags.
for col in LAB_COLS:
    df[col] = df[col].fillna(lab_medians[col])

cond_fill_cols = [c for c in df.columns if c not in LAB_COLS]
df[cond_fill_cols] = df[cond_fill_cols].fillna(0)

# -- Step 4: Log transform right-skewed labs ----------------------------------─
# Applied before scaling so StandardScaler sees more Gaussian distributions.
# log1p (log(1+x)) handles near-zero values safely.
# Inverse transform at inference time: expm1 (exp(x) - 1).
#
# CRITICAL: Only labs confirmed NOT already log-transformed upstream are listed here.
# Verified by checking that inverse-transformed values fall within expected ICU ranges.
# Labs with astronomical inverse values (sodium, bicarbonate etc.) were EXCLUDED
# because they were already transformed in the upstream cleaning pipeline.

LOG_TRANSFORM_COLS = [
    'glucose',      # skew=1.06, confirmed raw clinical units
    'creatinine',   # skew=2.42, confirmed raw clinical units
    'lactate',      # skew=3.10, confirmed raw clinical units
    'pt',           # skew=3.89, confirmed raw clinical units
    'ptt',          # skew=1.48, confirmed raw clinical units
    'bilirubin',    # skew=5.84, confirmed raw clinical units
    'ast',          # skew=4.53, confirmed raw clinical units
]

for col in LOG_TRANSFORM_COLS:
    if col in df.columns:
        df[col] = np.log1p(df[col])

print(f'Log transform applied to: {LOG_TRANSFORM_COLS}')
print(f'\nSkewness after log transform:')
for col in LOG_TRANSFORM_COLS:
    print(f'  {col:16s}: {df[col].skew():.3f}')

# -- Final cohort summary ------------------------------------------------------
print(f'\n{"=" * 60}')
print('FINAL COHORT')
print(f'{"=" * 60}')
print(f'   Total admissions:          {len(df):,}')
print(f'   Total columns:             {df.shape[1]}')
print(f'   Lab columns:               {len(LAB_COLS)}')
print(f'   Mortality rate:            {df["hospital_expire_flag"].mean():.1%}')
print(f'\n   Care unit distribution:')
icu_cols = [c for c in df.columns if c.startswith('icu_')]
for col, count in df[icu_cols].sum().sort_values(ascending=False).items():
    unit = col.replace('icu_', '')
    pct  = count / len(df)
    print(f'     {unit:10s}: {count:.0f} ({pct:.1%})')

df.head(3)

Excluded 7,054 remaining NICU patients
Dropped icu_NICU column -- all-zero after exclusion
Log transform applied to: ['glucose', 'creatinine', 'lactate', 'pt', 'ptt', 'bilirubin', 'ast']

Skewness after log transform:
  glucose         : 1.108
  creatinine      : 1.993
  lactate         : 1.810
  pt              : 3.155
  ptt             : 2.105
  bilirubin       : 3.909
  ast             : 2.712

FINAL COHORT
   Total admissions:          49,304
   Total columns:             86
   Lab columns:               23
   Mortality rate:            11.6%

   Care unit distribution:
     MICU      : 19525 (39.6%)
     CSRU      : 8580 (17.4%)
     SICU      : 8019 (16.3%)
     CCU       : 7198 (14.6%)
     TSICU     : 5982 (12.1%)


,hadm_id,subject_id,age,gender_bin,hospital_expire_flag,icu_CCU,icu_CSRU,icu_MICU,icu_SICU,icu_TSICU,...,pco2,ph,phosphate,platelets,po2,potassium,pt,ptt,sodium,wbc
0,165315,22,64.926812,0,0,0.0,0.0,1.0,0.0,0.0,...,29.0,7.47,3.7,259.0,287.0,4.4,2.595255,3.437208,140.0,5.1
1,152223,23,71.130191,1,0,0.0,1.0,0.0,0.0,0.0,...,41.0,7.39,3.2,95.0,370.0,3.6,2.917771,3.761200,140.0,7.6
2,124321,23,75.254799,1,0,0.0,0.0,0.0,1.0,0.0,...,42.0,7.40,4.2,208.0,286.0,3.5,2.587764,3.238678,133.0,14.8


## 4. Define Column Groups

Clearly labelled column groups for downstream use.

In [ ]:
# -- Column group definitions --------------------------------------------------
META_COLS = ['hadm_id', 'subject_id', 'hospital_expire_flag']

DEMOGRAPHIC_COLS = ['age', 'gender_bin']

ICU_COLS = [c for c in df.columns if c.startswith('icu_')]
LOS_COLS = [c for c in df.columns if c.startswith('los_')]
CCS_COLS = [c for c in df.columns if c.startswith('ccs_')]
ELIX_COLS = [c for c in df.columns if c.startswith('elix_')]

# All conditioning variables -- what you specify at generation/inference time
COND_COLS = DEMOGRAPHIC_COLS + ICU_COLS + LOS_COLS + CCS_COLS + ELIX_COLS

# Lab feature columns -- the generation target for cVAE
# Also usable as features or targets for any other downstream model
FEATURE_COLS = LAB_COLS

# All feature columns (conditioning + labs) for models that use everything
ALL_FEATURE_COLS = COND_COLS + FEATURE_COLS

# Sanity check
missing_labs = [c for c in FEATURE_COLS if c not in df.columns]
missing_cond = [c for c in COND_COLS    if c not in df.columns]
overlap      = set(FEATURE_COLS) & set(COND_COLS)

if missing_labs: print(f'WARNING: missing labs: {missing_labs}')
if missing_cond: print(f'WARNING: missing cond: {missing_cond}')
if overlap:      print(f'WARNING: overlap between FEATURE and COND: {overlap}')

print(f'Conditioning dims:     {len(COND_COLS)}')
print(f'Feature (lab) dims:    {len(FEATURE_COLS)}')
print(f'Total feature dims:    {len(ALL_FEATURE_COLS)}')
print(f'Log-transformed labs:  {LOG_TRANSFORM_COLS}')

Conditioning dims:     60
Feature (lab) dims:    23
Total feature dims:    83
Log-transformed labs:  ['glucose', 'creatinine', 'lactate', 'pt', 'ptt', 'bilirubin', 'ast']


## 5. Scale Features

In [ ]:
X_raw = df[FEATURE_COLS].values.astype(np.float32)
C_raw = df[COND_COLS].values.astype(np.float32)

# Fit scalers on full dataset
# Note: if splitting train/val for a specific model, refit x_scaler on train split only
x_scaler = StandardScaler()
c_scaler = StandardScaler()

X_scaled = x_scaler.fit_transform(X_raw)
C_scaled = c_scaler.fit_transform(C_raw)

# -- Inverse transform helper --------------------------------------------------
# Use this at inference time to convert scaled model outputs back to clinical units.
# Reverses operations in order: inverse scale first, then inverse log.

def inverse_transform_labs(scaled_array):
    """
    Convert scaled model output back to original clinical units.

    Args:
        scaled_array: numpy array or tensor of shape (n_samples, n_labs)
                      in StandardScaler space

    Returns:
        DataFrame with original clinical units, columns = FEATURE_COLS
    """
    if hasattr(scaled_array, 'cpu'):   # handle torch tensors
        scaled_array = scaled_array.cpu().numpy()

    # Step 1: inverse StandardScaler
    raw = x_scaler.inverse_transform(scaled_array)
    result = pd.DataFrame(raw, columns=FEATURE_COLS)

    # Step 2: inverse log1p for labs that were log-transformed
    for col in LOG_TRANSFORM_COLS:
        if col in result.columns:
            result[col] = np.expm1(result[col])

    return result

print('Scalers fitted.')
print(f'X_scaled shape: {X_scaled.shape}')
print(f'C_scaled shape: {C_scaled.shape}')
print(f'\nX_scaled stats (should be near mean=0, std=1 per column):')
print(f'  mean: {X_scaled.mean(axis=0).round(3)}')
print(f'  std:  {X_scaled.std(axis=0).round(3)}')

Scalers fitted.
X_scaled shape: (49304, 23)
C_scaled shape: (49304, 60)

X_scaled stats (should be near mean=0, std=1 per column):
  mean: [-0.  0. -0. -0.  0.  0.  0.  0.  0.  0. -0. -0.  0. -0.  0.  0. -0.  0.
 -0.  0.  0. -0. -0.]
  std:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 6. Save Outputs

In [ ]:
# -- Save unscaled feature matrix ----------------------------------------------
# Contains all columns including meta, conditioning, and lab features.
# Labs are log-transformed where applicable but NOT yet StandardScaler normalized.
# Use this for exploratory analysis or models that do their own scaling.
df.to_csv(f'{OUTPUT_DIR}lab_features.csv', index=False)
print(f'Saved lab_features.csv: {df.shape}')

# -- Save scaled feature matrix ------------------------------------------------
# Labs and conditioning variables are StandardScaler normalized.
# Use this as direct input to neural network models.
df_scaled = df[META_COLS].copy()
df_scaled[COND_COLS]    = C_scaled
df_scaled[FEATURE_COLS] = X_scaled
df_scaled.to_csv(f'{OUTPUT_DIR}lab_features_scaled.csv', index=False)
print(f'Saved lab_features_scaled.csv: {df_scaled.shape}')

# -- Save preprocessing metadata ----------------------------------------------─
# Contains everything needed to reproduce the preprocessing or inverse transform
# model outputs back to clinical units at inference time.
metadata = {
    # Scalers
    'x_scaler':            x_scaler,
    'c_scaler':            c_scaler,
    # Column definitions
    'feature_cols':        FEATURE_COLS,
    'cond_cols':           COND_COLS,
    'meta_cols':           META_COLS,
    'all_feature_cols':    ALL_FEATURE_COLS,
    'demographic_cols':    DEMOGRAPHIC_COLS,
    'icu_cols':            ICU_COLS,
    'los_cols':            LOS_COLS,
    'ccs_cols':            CCS_COLS,
    'elix_cols':           ELIX_COLS,
    # Transform info
    'log_transform_cols':  LOG_TRANSFORM_COLS,
    'lab_medians':         lab_medians.to_dict(),
    'lab_bounds':          LAB_BOUNDS,
    'lab_itemids':         LAB_ITEMIDS,
    # Cohort info
    'careunit_map':        CAREUNIT_MAP,
    'lab_window_start':    LAB_WINDOW_START,
    'lab_window_end':      LAB_WINDOW_END,
    'n_admissions':        len(df),
    'mortality_rate':      df['hospital_expire_flag'].mean(),
}

with open(f'{OUTPUT_DIR}preprocessing_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print(f'Saved preprocessing_metadata.pkl')

print(f'\nAll outputs saved to {OUTPUT_DIR}')

Saved lab_features.csv: (49304, 86)
Saved lab_features_scaled.csv: (49304, 86)
Saved preprocessing_metadata.pkl

All outputs saved to data/processed/


## 7. Usage Guide for Downstream Models

This section shows how to load and use the preprocessed data in any downstream model.

In [ ]:
# -- Loading the data in another notebook ------------------------------------─

# Option A -- load unscaled and apply your own scaling
# df = pd.read_csv('data/processed/lab_features.csv')

# Option B -- load pre-scaled for direct neural network input
# df_scaled = pd.read_csv('data/processed/lab_features_scaled.csv')

# Load metadata for column definitions and inverse transform
# with open('data/processed/preprocessing_metadata.pkl', 'rb') as f:
#     meta = pickle.load(f)
#
# Access column groups:
#   meta['feature_cols']       -- 23 lab columns
#   meta['cond_cols']          -- ~60 conditioning columns
#   meta['log_transform_cols'] -- labs that were log-transformed
#
# Inverse transform model outputs:
#   result_df = inverse_transform_labs(model_output_array)

# -- Example: train a mortality classifier ------------------------------------
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Features: conditioning + lab values
# Target:   hospital mortality
X = df[ALL_FEATURE_COLS].values
y = df['hospital_expire_flag'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict_proba(X_test)[:, 1]
auc    = roc_auc_score(y_test, y_pred)

print(f'Example downstream task -- Hospital Mortality Prediction')
print(f'  Features: {len(ALL_FEATURE_COLS)} ({len(COND_COLS)} conditioning + {len(FEATURE_COLS)} labs)')
print(f'  Train size: {X_train.shape[0]:,}  |  Test size: {X_test.shape[0]:,}')
print(f'  AUC: {auc:.3f}')
print()
print('This AUC serves as the REAL DATA baseline for TSTR evaluation.')
print('Compare against a model trained on synthetic cVAE data tested on this same test set.')

Example downstream task -- Hospital Mortality Prediction
  Features: 83 (60 conditioning + 23 labs)
  Train size: 39,443  |  Test size: 9,861
  AUC: 0.858

This AUC serves as the REAL DATA baseline for TSTR evaluation.
Compare against a model trained on synthetic cVAE data tested on this same test set.


In [ ]:
# -- Sanity check: verify inverse transform produces clinical units ------------─
# Run this to confirm the inverse_transform_labs function works correctly.
# Expected output should match known ICU reference ranges.

sample_scaled = X_scaled[:5]
sample_original = inverse_transform_labs(sample_scaled)

CLINICAL_REFERENCE = {
    'glucose':    (40,   500,  'mg/dL'),
    'sodium':     (125,  155,  'mEq/L'),
    'creatinine': (0.4,  15,   'mg/dL'),
    'ph':         (6.8,  7.8,  'units'),
    'lactate':    (0.1,  15,   'mmol/L'),
}

print('Inverse transform sanity check (first 5 patients):')
print(f'{"Lab":16s}  {"Min":>8s}  {"Max":>8s}  {"Expected range":20s}  {"Status"}')
print('-' * 75)
for lab, (lo, hi, unit) in CLINICAL_REFERENCE.items():
    actual_min = sample_original[lab].min()
    actual_max = sample_original[lab].max()
    # Allow generous range since 5 patients is a small sample
    ok = actual_min >= lo * 0.5 and actual_max <= hi * 3
    status = 'OK' if ok else 'WARNING -- check for double transform'
    print(f'{lab:16s}  {actual_min:8.2f}  {actual_max:8.2f}  {f"{lo}-{hi} {unit}":20s}  {status}')

Inverse transform sanity check (first 5 patients):
Lab                    Min       Max  Expected range        Status
---------------------------------------------------------------------------
glucose             100.00    378.00  40-500 mg/dL          OK
sodium              133.00    140.00  125-155 mEq/L         OK
creatinine            0.60      1.60  0.4-15 mg/dL          OK
ph                    7.39      7.47  6.8-7.8 units         OK
lactate               1.60      2.80  0.1-15 mmol/L         OK
